# Riyadh Natural-20 Autopilot / التشغيل التلقائي

**مرة واحدة فقط:** اختر T4 GPU ثم Runtime → Run all ووافق على Google Drive. يشغّل الدفتر عدة استراتيجيات تلقائياً لمدة تصل إلى 150 دقيقة، ويحفظ أفضل checkpoint بعد كل جولة، ويتوقف إذا حقق أقل من 20.

**One start only:** select T4 GPU, Run all, and approve Drive. The notebook cycles protected coordinate, hybrid, and CEM refinements for up to 150 minutes. Every method starts from the verified checkpoint and cannot replace it with a worse stock score.

> لا تعديل للمحاكي ولا RNG ولا trajectory injection. Free Colab availability and runtime are not guaranteed.

In [ ]:
# 1) GPU integrity check / فحص كرت الشاشة
import os, subprocess, sys
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout[:3000])
if gpu.returncode != 0:
    raise RuntimeError('GPU غير مفعّل. اختر Runtime > Change runtime type > GPU ثم أعد التشغيل.')

In [ ]:
# 2) Persistent checkpoints / حفظ دائم في Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/RiyadhNatural20'
os.makedirs(f'{DRIVE_ROOT}/actions', exist_ok=True)
print('Checkpoint folder:', DRIVE_ROOT)

In [ ]:
# 3) Official challenge + GPU runtime / المستودع الرسمي وبيئة GPU
import pathlib
if not pathlib.Path('/content/controls_challenge/.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/commaai/controls_challenge.git', '/content/controls_challenge'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu==1.24.4', 'pandas', 'tqdm', 'matplotlib', 'seaborn'], check=True)
os.chdir('/content/controls_challenge')
print('Repository ready:', os.getcwd())

In [ ]:
# 4) Fetch the original workers from the dedicated public branch / جلب العامل تلقائياً
import urllib.request
BASE = 'https://raw.githubusercontent.com/HadiRx/wasfa-project/controls-challenge-compute/compute'
for remote, local in [
    ('optimize_natural_cem.py', 'optimize_natural_cem.py'),
    ('optimize_natural_hybrid.py', 'optimize_natural_hybrid.py'),
    ('optimize_natural_coordinate.py', 'optimize_natural_coordinate.py'),
    ('natural_colab_worker.py', 'natural_colab_worker.py'),
    ('riyadh_pid.py', 'controllers/riyadh_pid.py')]:
    urllib.request.urlretrieve(f'{BASE}/{remote}', local)
print('Workers installed automatically.')

In [ ]:
# 5) Download official synthetic data once per fresh VM / تنزيل البيانات الرسمية
os.chdir('/content/controls_challenge')
if not pathlib.Path('data/00000.csv').exists():
    from tinyphysics import download_dataset
    download_dataset()
import torch  # preload Colab CUDA + cuDNN libraries before ONNX Runtime
import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    ort.preload_dlls()
providers = ort.get_available_providers()
print('ONNX providers:', providers)
if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError('CUDAExecutionProvider غير متاح؛ أعد تشغيل الـruntime ثم شغّل الخلايا من البداية.')

In [ ]:
# 6) Autopilot configuration / إعداد التشغيل التلقائي
START_SEGMENT = 0
END_SEGMENT = 1
TARGET_SCORE = 20.0
AUTOPILOT_BUDGET_MINUTES = 150
POPULATION = 512
ACTION_PATH = f'{DRIVE_ROOT}/actions/{START_SEGMENT:05d}.npy'
if not os.path.exists(ACTION_PATH):
    raise RuntimeError('لا يوجد checkpoint محفوظ. شغّل دفتر البداية أولاً.')
ROUNDS = [
    {'method':'coordinate', 'seed':2026, 'coord_iter':8, 'coord_spacings':'20,10,5,2,1', 'coord_deltas':'0.020,0.015,0.010,0.006,0.004'},
    {'method':'hybrid', 'seed':2031, 'hybrid_iter':12, 'spacings':'20,10,5,2,1', 'scales':'0.012,0.009,0.006,0.004,0.0025'},
    {'method':'cem', 'seed':2037, 'iterations':80, 'initial_std':0.012, 'rho':0.98},
    {'method':'hybrid', 'seed':2047, 'hybrid_iter':14, 'spacings':'10,5,2,1', 'scales':'0.008,0.005,0.003,0.0018'},
    {'method':'coordinate', 'seed':2053, 'coord_iter':10, 'coord_spacings':'10,5,2,1', 'coord_deltas':'0.012,0.008,0.005,0.003'},
    {'method':'cem', 'seed':2063, 'iterations':100, 'initial_std':0.007, 'rho':0.985},
]
print('Checkpoint:', ACTION_PATH)
print('Autopilot rounds:', len(ROUNDS))
print('Target / budget minutes:', TARGET_SCORE, AUTOPILOT_BUDGET_MINUTES)

In [ ]:
# 7) Protected multi-strategy autopilot / تشغيل تلقائي محمي
import time, json
deadline = time.time() + AUTOPILOT_BUDGET_MINUTES * 60
progress_path = f'{DRIVE_ROOT}/actions/progress.jsonl'

def latest_score():
    if not os.path.exists(progress_path):
        return None
    with open(progress_path, encoding='utf-8') as stream:
        rows = [json.loads(line) for line in stream if line.strip()]
    rows = [r for r in rows if r.get('segment') == f'{START_SEGMENT:05d}' and r.get('status') == 'completed']
    return float(rows[-1]['stock_cost']) if rows else None

print('AUTOPILOT START score=', latest_score(), flush=True)
for round_index, cfg in enumerate(ROUNDS, start=1):
    score = latest_score()
    if score is not None and score < TARGET_SCORE:
        print('TARGET ACHIEVED:', score, flush=True)
        break
    remaining = int((deadline - time.time()) / 60)
    if remaining < 5:
        print('Autopilot time budget reached safely.', flush=True)
        break
    cmd = [sys.executable, '-u', 'natural_colab_worker.py',
           '--data_dir', 'data', '--model_path', 'models/tinyphysics.onnx',
           '--out_dir', f'{DRIVE_ROOT}/actions',
           '--start', str(START_SEGMENT), '--end', str(END_SEGMENT),
           '--time_budget_minutes', str(min(55, remaining)),
           '--provider', 'CUDAExecutionProvider', '--population', str(POPULATION),
           '--method', cfg['method'], '--seed', str(cfg['seed']), '--refine']
    if cfg['method'] == 'coordinate':
        cmd += ['--coordinate_iterations', str(cfg['coord_iter']),
                '--coordinate_spacings', cfg['coord_spacings'],
                '--coordinate_deltas', cfg['coord_deltas']]
    elif cfg['method'] == 'hybrid':
        cmd += ['--hybrid_iterations', str(cfg['hybrid_iter']),
                '--spacings', cfg['spacings'], '--scales', cfg['scales']]
    else:
        cmd += ['--iterations', str(cfg['iterations']),
                '--initial_std', str(cfg['initial_std']), '--rho', str(cfg['rho']),
                '--min_std', '0.0008', '--momentum', '0.20']
    print(f'ROUND {round_index}/{len(ROUNDS)} method={cfg["method"]} score={score} remaining={remaining}m', flush=True)
    result = subprocess.run(cmd)
    print('ROUND RETURN CODE:', result.returncode, 'new score=', latest_score(), flush=True)
    if result.returncode != 0:
        raise RuntimeError(f'Autopilot round {round_index} failed.')
print('AUTOPILOT DONE best verified score=', latest_score(), flush=True)

In [ ]:
# 8) Progress summary / ملخص التقدم
import json, glob, numpy as np
records = []
log_path = f'{DRIVE_ROOT}/actions/progress.jsonl'
if os.path.exists(log_path):
    with open(log_path, encoding='utf-8') as stream:
        records = [json.loads(line) for line in stream if line.strip()]
ok = [r for r in records if r.get('status') == 'completed']
latest = {}
for record in ok:
    latest[record['segment']] = record
costs = [r['stock_cost'] for r in latest.values()]
print('Saved action files:', len(glob.glob(f'{DRIVE_ROOT}/actions/*.npy')))
print('Verified segments:', len(costs))
if costs:
    print('Verified stock mean:', float(np.mean(costs)))
    print('Best / worst:', float(np.min(costs)), float(np.max(costs)))
    print('Target gate:', 'PASS < 20' if np.mean(costs) < 20 else 'NOT YET')

## قرار التوسعة / Scale decision

لا توسّع إلى 5000 فقط لأن الخلايا تعمل. راجع متوسط `stock_cost` وسرعة segment الواحد أولاً. إذا كان المتوسط بعيداً عن 20 نحتاج تحسين الخوارزمية قبل استهلاك جلسات إضافية.

Do not scale to 5000 merely because the worker runs. Review verified stock cost and per-segment throughput first. A final result is valid only after a stock 5000-segment evaluation.